# 02 — Análise exploratória (EDA) e redução de dimensionalidade

**Fase 2 do enunciado — EDA:**
1. Estatística descritiva e visualizações (histogramas, gráficos de relação) para investigar padrões, outliers e associações.
2. Pelo menos um método linear e um não linear de redução de dimensionalidade → **PCA** e **UMAP**.
3. EDA feita **só com dados de desenvolvimento**, separada do teste final; discutir os primeiros *insights*.

**Que dados entram (decidido na Fase 1):**

| Parte | Dados | Porquê |
|---|---|---|
| Regressão | `regressao_dev.pkl` (2013-01 a 2014-12) | É o treino da regressão (as 37 colunas do *bureau* só estão completas a partir de 2013) |
| Classificação | `classificacao_treino.pkl` (36 meses, 2007-06 a 2011-12) | É o treino da classificação; análises com o desfecho só aqui |
| Teste (2015) | **não é aberto** | Só na Parte 2, uma vez |

**Regras:** este notebook só **descreve** os dados — não remove nem imputa nada (isso é limpeza/pré-processamento). Gráficos com muitos pontos usam amostras reprodutíveis (semente 42), nunca as primeiras linhas. As figuras são gravadas em `reports/figures/` para o relatório.


In [ ]:
# Setup
import sys
from pathlib import Path

try:
    import lending
except ModuleNotFoundError:                      # plano B se não fizeste pip install -e .
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lending import config
from lending.storage import PickleStore
from lending.eda import DescriptiveAnalysis, EDAPlotter
from lending.dimred import DimensionalityReducer

pd.set_option("display.max_columns", None)
plotter = EDAPlotter()            # aplica o estilo do projeto e grava em reports/figures/


## Colunas usadas

As listas vêm da tabela de elegibilidade (`docs/01_formulacao_problema/elegibilidade_colunas.md`): só entram colunas ✅ para a tarefa. Os nomes ficam **aqui**, não dentro das classes (nota do professor sobre valores *hardcoded*).

- `fico_range_high` não entra: é sempre `fico_range_low` + 4, por isso não acrescenta informação.
- `*_LOG`: variáveis com cauda longa, mostradas em escala log.
- `*_CLIP`: variáveis com alguns valores extremos; a cauda é escondida **só no gráfico**.
- Se alguma coluna não existir nos ficheiros do split, é retirada automaticamente (aviso abaixo).


In [ ]:
# --- Regressão (pré-pricing: sem grade, sub_grade, installment, verification_status) ---
REG_TARGET = "int_rate"
REG_NUM = ["loan_amnt", "annual_inc", "dti", "fico_range_low", "revol_util", "revol_bal",
           "open_acc", "total_acc", "inq_last_6mths", "delinq_2yrs", "pub_rec", "mort_acc",
           "tot_cur_bal", "tot_hi_cred_lim", "bc_util", "bc_open_to_buy",
           "mo_sin_old_rev_tl_op", "num_actv_rev_tl", "pct_tl_nvr_dlq"]
REG_CAT = ["term", "purpose", "home_ownership", "emp_length"]
REG_LOG = ["annual_inc", "revol_bal", "tot_cur_bal", "tot_hi_cred_lim", "bc_open_to_buy"]
REG_CLIP = ["dti", "revol_util", "bc_util", "open_acc", "total_acc", "inq_last_6mths"]

# --- Classificação (na concessão: pricing permitido; sem as 37 colunas que só existem desde 2012) ---
CLS_NUM = ["loan_amnt", "annual_inc", "dti", "fico_range_low", "revol_util", "revol_bal",
           "open_acc", "total_acc", "inq_last_6mths", "delinq_2yrs", "pub_rec",
           "int_rate", "installment"]
CLS_CAT = ["grade", "purpose", "home_ownership", "verification_status", "emp_length"]
CLS_LOG = ["annual_inc", "revol_bal"]
CLS_CLIP = ["dti", "revol_util", "open_acc", "total_acc", "inq_last_6mths"]

GRADE_ORDER = list("ABCDEFG")


def keep_existing(cols, df, name):
    """Devolve só as colunas que existem em df e avisa das que faltam."""
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"[{name}] colunas em falta (ignoradas): {missing}")
    return [c for c in cols if c in df.columns]


## 1. Carregar os dados de desenvolvimento

In [ ]:
reg = PickleStore.load(config.REG_DEV_PKL, expected_type=pd.DataFrame)
cls = PickleStore.load(config.CLS_TRAIN_PKL, expected_type=pd.DataFrame)

# Alvo binário auxiliar para taxas de default (1 = Default, 0 = Good standing)
cls["is_default"] = (cls[config.CLASS_COL] == "Default").astype(int)

REG_NUM = keep_existing(REG_NUM, reg, "regressão")
REG_CAT = keep_existing(REG_CAT, reg, "regressão")
CLS_NUM = keep_existing(CLS_NUM, cls, "classificação")
CLS_CAT = keep_existing(CLS_CAT, cls, "classificação")

print(f"Regressão (dev):      {reg.shape[0]:,} linhas x {reg.shape[1]} colunas")
print(f"Classificação (treino): {cls.shape[0]:,} linhas x {cls.shape[1]} colunas")


---
# Parte A — Regressão (`int_rate`, 2013–2014)

## A.1 Estatística descritiva, valores em falta e outliers

In [ ]:
reg_desc = DescriptiveAnalysis(reg)
reg_desc.summary(REG_NUM + [REG_TARGET])


In [ ]:
plotter.missing_bar(reg, REG_NUM + REG_CAT, filename="reg_missing.png");

In [ ]:
# Outliers pelas barreiras de Tukey (1,5 x IQR) — só diagnóstico, nada é removido
reg_desc.outliers(REG_NUM + [REG_TARGET])


## A.2 Distribuições

In [ ]:
plotter.histograms(reg, [REG_TARGET] + REG_NUM, log_cols=REG_LOG, clip_cols=REG_CLIP,
                   title="Regressão (2013–2014): distribuições", filename="reg_histogramas.png");

## A.3 Relações com o alvo (`int_rate`)

In [ ]:
# Relação de int_rate com as variáveis de risco mais diretas (densidade + mediana)
for x in ["fico_range_low", "dti", "revol_util", "annual_inc"]:
    if x in reg.columns:
        plotter.hexbin(reg, x, REG_TARGET, filename=f"reg_{x}_vs_rate.png")


In [ ]:
for cat in REG_CAT:
    plotter.box_by_category(reg, REG_TARGET, cat, filename=f"reg_rate_por_{cat}.png")


In [ ]:
# Tabelas de apoio: int_rate médio por categoria (valores para citar no relatório)
for cat in REG_CAT:
    display(reg_desc.by_category(cat, REG_TARGET, min_count=100))


In [ ]:
plotter.correlation_heatmap(reg, REG_NUM + [REG_TARGET], method="spearman",
                            filename="reg_correlacao.png");

## A.4 PCA e UMAP (regressão)

O pré-processamento (log → imputação pela mediana → padronização) e o PCA são **ajustados só nestes dados de desenvolvimento**, dentro de um `Pipeline`. O UMAP usa uma amostra reprodutível de 20 000 empréstimos (é lento). Os pontos são coloridos por `int_rate`: se as cores se organizarem no mapa, as variáveis contêm informação sobre a taxa.

In [ ]:
reg_dr = DimensionalityReducer(REG_NUM, log_cols=REG_LOG, sample_size=20_000).fit_pca(reg)
print("Componentes para 80% da variância:", reg_dr.n_components_for(0.8))
reg_dr.plot_scree(filename="reg_pca_scree.png");


In [ ]:
# O que mede cada componente (pesos das variáveis)
reg_dr.loadings(n_components=3, top=10)


In [ ]:
reg_sample = reg_dr.sampler.sample(reg, 20_000)
reg_dr.plot_embedding(reg_dr.pca_coordinates(reg_sample), reg_sample[REG_TARGET], kind="continuous",
                      label="int_rate", title="PCA (PC1 x PC2) — cor: int_rate",
                      xlabel="PC1", ylabel="PC2", filename="reg_pca_2d.png");


In [ ]:
reg_umap_sample = reg_dr.fit_umap(reg)       # pode demorar 1–3 minutos
reg_dr.plot_embedding(reg_dr.umap_embedding, reg_umap_sample[REG_TARGET], kind="continuous",
                      label="int_rate", title="UMAP — cor: int_rate",
                      xlabel="UMAP 1", ylabel="UMAP 2", filename="reg_umap_2d.png");


---
# Parte B — Classificação (treino: 36 meses, 2007–2011)

## B.1 Distribuição das classes

São só dois números, por isso fica uma tabela e não um gráfico. A classe *At risk* tem 0 casos no treino (ver Fase 1).

In [ ]:
class_table = (cls[config.CLASS_COL].value_counts()
               .reindex(config.CLASS_ORDER)
               .to_frame("n")
               .assign(pct=lambda t: (100 * t["n"] / t["n"].sum()).round(1)))
class_table


## B.2 Estatística descritiva e outliers

In [ ]:
cls_desc = DescriptiveAnalysis(cls)
cls_desc.summary(CLS_NUM)


In [ ]:
cls_desc.outliers(CLS_NUM)

In [ ]:
plotter.histograms(cls, CLS_NUM, log_cols=CLS_LOG, clip_cols=CLS_CLIP,
                   title="Classificação (treino 2007–2011): distribuições",
                   filename="cls_histogramas.png");

## B.3 Relações com o desfecho

Taxa de default por categoria e por decil, e distribuição de cada variável por classe (*raincloud*: forma + caixa + pontos).

In [ ]:
if "grade" in cls.columns:
    plotter.rate_by_category(cls, "grade", "is_default", order=GRADE_ORDER, xlabel="% default",
                             filename="cls_default_por_grade.png")
    display(cls_desc.by_category("grade", "is_default"))


In [ ]:
for cat in [c for c in CLS_CAT if c != "grade"]:
    plotter.rate_by_category(cls, cat, "is_default", xlabel="% default",
                             filename=f"cls_default_por_{cat}.png")


In [ ]:
for col in ["dti", "fico_range_low", "int_rate", "revol_util", "annual_inc"]:
    if col in cls.columns:
        plotter.raincloud(cls, col, config.CLASS_COL, order=config.CLASS_ORDER,
                          log=col in CLS_LOG, filename=f"cls_raincloud_{col}.png")


In [ ]:
for col in ["dti", "fico_range_low", "annual_inc", "revol_util"]:
    if col in cls.columns:
        plotter.rate_by_bins(cls, col, "is_default", filename=f"cls_default_por_decil_{col}.png")


In [ ]:
plotter.correlation_heatmap(cls, CLS_NUM + ["is_default"], method="spearman",
                            filename="cls_correlacao.png");

## B.4 PCA e UMAP (classificação)

Pontos coloridos pela classe, sempre com a mesma ordem de cores (`CLASS_ORDER`: Good standing azul, Default laranja). A amostra do UMAP é estratificada pela classe, para manter a proporção 87,7% / 12,3%.

In [ ]:
cls_dr = DimensionalityReducer(CLS_NUM, log_cols=CLS_LOG, sample_size=15_000).fit_pca(cls)
print("Componentes para 80% da variância:", cls_dr.n_components_for(0.8))
cls_dr.plot_scree(filename="cls_pca_scree.png")
cls_dr.loadings(n_components=3, top=10)


In [ ]:
cls_sample = cls_dr.sampler.sample(cls, 15_000, stratify=config.CLASS_COL)
cls_dr.plot_embedding(cls_dr.pca_coordinates(cls_sample), cls_sample[config.CLASS_COL],
                      order=config.CLASS_ORDER, label="classe",
                      title="PCA (PC1 x PC2) — cor: classe", xlabel="PC1", ylabel="PC2",
                      filename="cls_pca_2d.png");


In [ ]:
cls_umap_sample = cls_dr.fit_umap(cls, stratify=config.CLASS_COL)
cls_dr.plot_embedding(cls_dr.umap_embedding, cls_umap_sample[config.CLASS_COL],
                      order=config.CLASS_ORDER, label="classe", title="UMAP — cor: classe",
                      xlabel="UMAP 1", ylabel="UMAP 2", filename="cls_umap_2d.png");


---
# Parte C — Primeiros *insights* (para a secção 2b do relatório)

Preencher depois de correr o notebook com os dados reais. Para cada ponto: **o que se vê** (com número) → **o que significa** → **o que muda nas próximas fases**.

**Regressão**
1. Forma de `int_rate` (simétrica? multimodal por prazo?) → …
2. Variáveis com relação mais forte com a taxa (hexbin, correlação) → candidatas a features.
3. Diferenças por `term` e `purpose` → …
4. Variáveis muito assimétricas / outliers → transformação log e escalamento robusto no pipeline.
5. Pares muito correlacionados (ex.: `revol_util` e `bc_util`) → redundância; ter em conta na seleção de features.

**Classificação**
1. Desequilíbrio 87,7% / 12,3% → métricas *macro*, balanceamento só nos folds de treino.
2. Taxa de default por `grade` (monotónica?) → o `grade` é uma *baseline* forte (critério de sucesso).
3. `dti`, `fico`, `int_rate` por classe → separação parcial ou forte?
4. Relações não lineares nos decis → modelos não lineares (árvores, ensembles) podem ganhar.

**PCA vs UMAP**
1. Quantas componentes para 80% → informação muito espalhada ou concentrada?
2. O que medem PC1 e PC2 (pesos) → …
3. As classes / a taxa separam-se no mapa? Mais no UMAP do que no PCA? → indica relações não lineares.
